# SFSC 2026 — V5 fusion: V4's pipeline on averaged UNet maps, optional level-augmented scorer (Colab CPU)

V4 unchanged except two switches (environment variables, or edit Cell 01):

| switch | meaning |
|---|---|
| `SFSC_MAPS` | UNet runs whose out-of-fold and test maps are averaged. `sfsc_unet_v3` = V4 exactly; step 3 adds `sfsc_v5/s3_unetpp` |
| `SFSC_AUG=1` | step 1: the keep-scorer also trains on candidates at 7 more hysteresis levels (0.15-0.70); exported candidates stay V4's |

Inputs (read only): V3 run, V4 detections (`sfsc_fusion_v4/detections`), extra map runs. Output: `outputs/sfsc_v5/fusion_<tag>/`
with `selection_v5.csv`, `image_stats.npz` (per-image PQ parts of every variant, for paired comparisons across runs) and
`test_export/submission_<variant>.csv`.

**Run:** CPU runtime (High-RAM). `Runtime -> Run all`. No GPU needed.


## Cell 01 — Drive access, switches, inputs (read only)


In [ ]:
from pathlib import Path
import os, shutil, subprocess

IN_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG"))
DRIVE_MODE = os.environ.get("SFSC_DRIVE_MODE", "mount" if IN_COLAB else "local")   # mount | rclone | local
RCLONE_REMOTE = os.environ.get("SFSC_RCLONE_REMOTE", "gdrive")
MAP_RUNS = [s.strip() for s in os.environ.get("SFSC_MAPS", "sfsc_unet_v3").split(",") if s.strip()]
AUG = os.environ.get("SFSC_AUG", "0") == "1"
assert MAP_RUNS[0] == "sfsc_unet_v3", "the first map run must be V3 (geometry, recipe and folds come from it)"
TAG = "+".join(Path(r).name for r in MAP_RUNS) + ("_aug" if AUG else "")
LOCAL_ROOT = Path(os.environ.get("SFSC_LOCAL_ROOT", "/content/sfsc_v5_fusion" if IN_COLAB else "/tmp/sfsc_v5_fusion"))
LOCAL_DATA = LOCAL_ROOT / "dataset"
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

def rclone(*args, check=True):
    return subprocess.run(["rclone", *args, "--transfers", "16", "--checkers", "16"], check=check)

if DRIVE_MODE == "mount":
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/SFSC2026")
elif DRIVE_MODE == "rclone":
    if shutil.which("rclone") is None:
        subprocess.run("curl -fsSL https://rclone.org/install.sh | bash", shell=True, check=True)
    DRIVE_ROOT = LOCAL_ROOT / "drive_mirror"
    remote = f"{RCLONE_REMOTE}:SFSC2026"
    if not os.environ.get("SFSC_DATASET"):
        rclone("copy", f"{remote}/data", str(DRIVE_ROOT / "data"), "--include", "/sfsc2026-data.zip")
    rclone("copy", f"{remote}/outputs/sfsc_unet_v3", str(DRIVE_ROOT / "outputs" / "sfsc_unet_v3"),
           "--include", "/{folds_v3.csv,frozen_recipe_v3.json}", "--include", "/fold*/{config,done}.json",
           "--include", "/oof/**", "--include", "/test_probs/**")
    for r in MAP_RUNS[1:]:
        rclone("copy", f"{remote}/outputs/{r}", str(DRIVE_ROOT / "outputs" / r), "--include", "/oof/**",
               "--include", "/test_probs/**", "--include", "/fold*/done.json")
    rclone("copy", f"{remote}/outputs/sfsc_fusion_v4/detections", str(DRIVE_ROOT / "outputs" / "sfsc_fusion_v4" / "detections"))
else:
    DRIVE_ROOT = Path(os.environ["SFSC_DRIVE_ROOT"])

DATA_ZIP = DRIVE_ROOT / "data" / "sfsc2026-data.zip"
DRIVE_EXTRACTED = DRIVE_ROOT / "data" / "SFSCDataset"
V3_RUN = DRIVE_ROOT / "outputs" / "sfsc_unet_v3"
MAP_DIRS = [DRIVE_ROOT / "outputs" / r for r in MAP_RUNS]
DET_DIR = DRIVE_ROOT / "outputs" / "sfsc_fusion_v4" / "detections"
V4_RUN_BASE = DRIVE_ROOT / "outputs" / "sfsc_v5" / f"fusion_{TAG}"

def push():
    if DRIVE_MODE == "rclone":
        rclone("copy", str(RUN), f"{RCLONE_REMOTE}:SFSC2026/outputs/sfsc_v5/{RUN.name}", check=False)

for _needed in ("folds_v3.csv", "frozen_recipe_v3.json"):
    if not (V3_RUN / _needed).is_file():
        raise FileNotFoundError(f"{V3_RUN / _needed} not found.")
print("Map runs averaged:", MAP_RUNS, "| level-augmented scorer:", AUG, "| output:", V4_RUN_BASE)


## Cell 02 — Dependencies (no detector library needed)


In [ ]:
import importlib.util, subprocess, sys

REQUIRED = {
    "sklearn": "scikit-learn>=1.3",
    "pycocotools": "pycocotools>=2.0.7",
    "cv2": "opencv-python-headless>=4.8",
    "scipy": "scipy>=1.10",
    "pandas": "pandas>=2.0",
    "tqdm": "tqdm>=4.66",
}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
import sklearn
print("sklearn", sklearn.__version__)


## Cell 03 — Imports, configuration, settings read from V3, helpers


In [ ]:
import copy, gc, hashlib, json, math, multiprocessing as mp, pickle, platform
import random, re, shutil, subprocess, sys, tempfile, time, warnings, zipfile
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from dataclasses import dataclass, asdict
from functools import lru_cache

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
from pycocotools import mask as mask_api
from sklearn.ensemble import HistGradientBoostingRegressor
from tqdm.auto import tqdm

@dataclass
class Config:
    # ---- run control ----
    debug_run: bool = False                  # True = quick pipeline check (separate *_debug folder)
    local_dataset_override: str | None = os.environ.get("SFSC_DATASET") or None
    export_test: bool = True
    skip_empty_records: bool = True          # the organizers' scorer never sees annotator sets without filaments

    # ---- detector: the second model family ----
    det_weights: str = "yolo11m-seg.pt"
    det_imgsz: int = 1280
    det_epochs: int = 40
    det_batch: int = 4
    det_cls_gain: float = 1.5                # classification-loss weight (Ultralytics default 0.5)
    det_workers: int = 8
    det_save_every: int = 2                  # copy the resume checkpoint to Drive every N epochs
    det_folds: tuple = (0, 1, 2, 3)
    det_conf_floor: float = 0.05             # lowest detection score kept as evidence
    det_nms_iou: float = 0.6
    det_max_det: int = 200

    # ---- fusion ----
    det_min_area: int = 32                   # px, smaller detections are ignored
    det_link_frac: float = 0.2               # a detection supports a candidate if they share this fraction of the smaller one
    det_only_floor: float = 0.10             # lowest score for a detection to create a new candidate
    det_only_max_claimed: float = 0.30       # ...and at most this fraction of it may already be covered
    det_grow: int = 4                        # px the detection is grown before UNet pixels are claimed inside it
    det_claim_t: float = 0.20                # UNet probability needed for a claimed pixel
    det_claim_min_frac: float = 0.50         # use UNet pixels if they cover this fraction of the detection, else its own mask
    det_only_min_area: int = 64

    # ---- selection ----
    min_gain: float = 0.003                  # a V4 variant must beat re-scored V3 by this out-of-fold PQ...
    min_fold_wins: int = 3                   # ...and win in at least this many folds
    bootstrap: int = 2000
    cpu_pool_workers: int | None = None
    aug_levels: tuple = (0.15, 0.20, 0.25, 0.30, 0.50, 0.60, 0.70)   # step 1: extra t_low levels (V4: 0.40)
    run_diagnostics: bool = os.environ.get("SFSC_DIAG", "0") == "1"

CFG = Config()
if CFG.debug_run:
    CFG.det_folds, CFG.det_epochs = (0,), 2
DEBUG_LIMIT = 24 if CFG.debug_run else None

# ---- settings that must match the V3 run are read from it, not retyped ----
_cfg_files = sorted(V3_RUN.glob("fold*/config.json"))
if not _cfg_files:
    raise FileNotFoundError(f"No fold*/config.json under {V3_RUN}")
V3CFG = json.loads(_cfg_files[0].read_text())["config"]
for _k in ("seed", "n_folds", "disk_radius_fraction", "radial_bins", "flat_median_kernel", "flat_std", "contact_radius",
           "connectivity", "fill_holes", "min_core_area", "contact_assign_radius", "scorer_prefilter_max",
           "scorer_prefilter_area"):
    setattr(CFG, _k, V3CFG[_k])
CFG.expected_shape = tuple(int(x) for x in V3CFG["expected_shape"])
CFG.scorer_taus = tuple(float(t) for t in V3CFG["scorer_taus"])

RUN = V4_RUN_BASE
RUN.mkdir(parents=True, exist_ok=True)
random.seed(CFG.seed); np.random.seed(CFG.seed)
cv2.setNumThreads(0)

CPU_COUNT = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 2)
CPU_POOL = max(2, CPU_COUNT - 2) if CFG.cpu_pool_workers is None else CFG.cpu_pool_workers

def hash_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for part in iter(lambda: f.read(1 << 20), b""):
            h.update(part)
    return h.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), default=str).encode()).hexdigest()

def write_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp = tempfile.mkstemp(prefix=".json_", dir=str(path.parent))
    try:
        with os.fdopen(fd, "w") as f:
            json.dump(obj, f, indent=2, default=str)
        os.replace(temp, path)
    finally:
        Path(temp).unlink(missing_ok=True)

def save_npy_atomic(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp = tempfile.mkstemp(prefix=".npy_", dir=str(path.parent))
    try:
        with os.fdopen(fd, "wb") as f:
            np.save(f, value, allow_pickle=False)
        os.replace(temp, path)
    finally:
        Path(temp).unlink(missing_ok=True)

def copy_atomic(src, dst):
    """Copy unless a file of the same size is already there."""
    src, dst = Path(src), Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.is_file() and dst.stat().st_size == src.stat().st_size:
        return
    copy_force(src, dst)

def copy_force(src, dst):
    """Always copy (checkpoints of the same size differ in content)."""
    src, dst = Path(src), Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    temp = dst.with_suffix(dst.suffix + ".part")
    shutil.copyfile(src, temp)
    os.replace(temp, dst)

def save_pickle(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".part")
    with open(temp, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(temp, path)

print("CPU cores:", CPU_COUNT, "| CPU scoring pool:", CPU_POOL)
print("Run folder:", RUN, "| debug_run =", CFG.debug_run)
print("From V3: seed", CFG.seed, "| image", CFG.expected_shape, "| disk radius fraction", CFG.disk_radius_fraction)


## Cell 04 — Stage the dataset (V4, unchanged)


In [ ]:
ANNOTATION_BASENAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
LOCAL_ZIP = LOCAL_ROOT / "sfsc_dataset.zip"

def locate_dataset(root):
    root = Path(root)
    hits = sorted(root.rglob(ANNOTATION_BASENAME)) if root.is_dir() else []
    if len(hits) != 1:
        raise FileNotFoundError(f"Expected one {ANNOTATION_BASENAME} under {root}; found {len(hits)}")
    ann = hits[0]
    train_dir = ann.parent / "train_images"
    test_dir = ann.parent.parent / "test" / "test_images"
    if not (train_dir.is_dir() and test_dir.is_dir()):
        raise FileNotFoundError(f"Missing train_images or test_images near {ann}")
    return ann, train_dir, test_dir

def image_paths(folder):
    return {p.name: p for p in Path(folder).iterdir()
            if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg")}

def staged_and_ready(root):
    try:
        a, t, s = locate_dataset(root)
        return a.stat().st_size > 1000 and len(image_paths(t)) > 0 and len(image_paths(s)) > 0
    except (FileNotFoundError, OSError):
        return False

def safe_extract(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    resolved_root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in tqdm(archive.infolist(), desc="Extract dataset"):
            if member.is_dir():
                continue
            dest = (destination / member.filename).resolve()
            if not dest.is_relative_to(resolved_root):
                raise ValueError(f"Unsafe ZIP member: {member.filename}")
            if dest.is_file() and dest.stat().st_size == member.file_size:
                continue
            dest.parent.mkdir(parents=True, exist_ok=True)
            archive.extract(member, destination)

if CFG.local_dataset_override:
    DATA_ROOT = Path(CFG.local_dataset_override)
    if not staged_and_ready(DATA_ROOT):
        raise FileNotFoundError(f"Dataset override not ready: {DATA_ROOT}")
elif staged_and_ready(LOCAL_DATA):
    DATA_ROOT = LOCAL_DATA
else:
    DATA_ROOT = LOCAL_DATA
    if DATA_ZIP.is_file():
        print("Copying ZIP to local SSD once per runtime...")
        copy_atomic(DATA_ZIP, LOCAL_ZIP)
        safe_extract(LOCAL_ZIP, DATA_ROOT)
    elif DRIVE_EXTRACTED.is_dir():
        print("Copying extracted dataset folder from Drive to local SSD...")
        shutil.copytree(DRIVE_EXTRACTED, DATA_ROOT, dirs_exist_ok=True)
    else:
        raise FileNotFoundError(f"Expected {DATA_ZIP} or {DRIVE_EXTRACTED}. Or set CFG.local_dataset_override.")
if not staged_and_ready(DATA_ROOT):
    raise RuntimeError("Local staging incomplete. Check ZIP contents and free disk space.")

ANN_PATH, TRAIN_DIR, TEST_DIR = locate_dataset(DATA_ROOT)
TRAIN_PATHS, TEST_PATHS = image_paths(TRAIN_DIR), image_paths(TEST_DIR)
for label, paths in (("train", TRAIN_PATHS), ("test", TEST_PATHS)):
    if len({Path(n).stem for n in paths}) != len(paths):
        raise ValueError(f"Duplicate {label} image stems; IDs would collide.")

def read_gray(path):
    with Image.open(path) as im:
        if im.mode == "L":
            return np.asarray(im, dtype=np.uint8).copy()
        if im.mode != "RGB":
            raise ValueError(f"Unexpected image mode {im.mode}: {path}")
        arr = np.asarray(im, dtype=np.uint8)
        d1 = np.abs(arr[:, :, 0].astype(np.int16) - arr[:, :, 1])
        d2 = np.abs(arr[:, :, 0].astype(np.int16) - arr[:, :, 2])
        if d1.max() > 2 or d2.max() > 2:
            raise ValueError(f"Non-grayscale RGB image: {path}")
        return arr[:, :, 0].copy()

RAW = json.loads(ANN_PATH.read_text())
ANN_SHA = hash_file(ANN_PATH)
print("Annotation JSON:", ANN_PATH, "| sha256", ANN_SHA[:16])
print("JPEGs:", len(TRAIN_PATHS), "train /", len(TEST_PATHS), "test | free local GiB:",
      round(shutil.disk_usage(LOCAL_ROOT).free / 2**30, 1))


## Cell 05 — COCO parsing, record index (V4, unchanged)


In [ ]:
H, W = CFG.expected_shape

def native_rle(rle):
    counts = rle["counts"]
    if isinstance(counts, str):
        counts = counts.encode("ascii")
    return {"size": [int(x) for x in rle["size"]], "counts": counts}

def json_rle(rle):
    counts = rle["counts"]
    return {"size": [int(x) for x in rle["size"]],
            "counts": counts.decode("ascii") if isinstance(counts, bytes) else counts}

def decode_mask(rle):
    a = mask_api.decode(native_rle(rle))
    return np.any(a, axis=2) if a.ndim == 3 else (a > 0)

def segmentation_to_rle(seg, h, w):
    if isinstance(seg, dict):
        if list(seg.get("size", [h, w])) != [h, w]:
            raise ValueError("Annotation RLE shape mismatch")
        rle = (mask_api.frPyObjects(seg, h, w) if isinstance(seg.get("counts"), list)
               else native_rle({"size": [h, w], "counts": seg["counts"]}))
    elif isinstance(seg, list) and len(seg):
        polys = seg if isinstance(seg[0], (list, tuple)) else [seg]
        if any(len(p) < 6 or len(p) % 2 for p in polys):
            raise ValueError("Malformed polygon segmentation")
        rle = mask_api.merge(mask_api.frPyObjects(polys, h, w))
    else:
        raise ValueError("Unsupported empty/malformed segmentation")
    return json_rle(rle)

STEMS = defaultdict(list)
for _n in TRAIN_PATHS:
    STEMS[Path(_n).stem].append(_n)

def canonical_filename(s):
    name = Path(str(s)).name
    if name in TRAIN_PATHS:
        return name
    cands = STEMS.get(Path(name).stem, [])
    if len(cands) == 1:
        return cands[0]
    raise ValueError(f"Unresolvable annotated JPEG: {s}")

IMAGES_BY_ID = {}
for im in RAW["images"]:
    rid = str(im["id"])
    if rid in IMAGES_BY_ID:
        raise ValueError(f"Duplicate image record {rid}")
    if (int(im["height"]), int(im["width"])) != (H, W):
        raise ValueError(f"Unexpected dimensions for record {rid}")
    IMAGES_BY_ID[rid] = im

ANNOTATIONS_BY_ID = defaultdict(list)
_seen = set()
for a in RAW["annotations"]:
    aid, rid = str(a["id"]), str(a["image_id"])
    if aid in _seen or rid not in IMAGES_BY_ID:
        raise ValueError(f"Duplicate annotation or missing image: {aid} / {rid}")
    if int(a.get("iscrowd", 0)):
        raise ValueError(f"Unexpected crowd annotation {aid}")
    if int(a.get("category_id", 1)) not in (1, 2, 3, 4):
        raise ValueError(f"Unexpected category {a.get('category_id')} in {aid}")
    _seen.add(aid)
    ANNOTATIONS_BY_ID[rid].append(a)

RECORD_IDS_BY_FILE = defaultdict(list)
for rid, im in IMAGES_BY_ID.items():
    RECORD_IDS_BY_FILE[canonical_filename(im["file_name"])].append(rid)
for k in RECORD_IDS_BY_FILE:
    RECORD_IDS_BY_FILE[k].sort()
UNIQUE_FILES = sorted(RECORD_IDS_BY_FILE)

FILE_HASHES = {}
with ThreadPoolExecutor(max_workers=8) as pool:
    for name, digest in zip(UNIQUE_FILES, pool.map(lambda n: hash_file(TRAIN_PATHS[n]), UNIQUE_FILES)):
        FILE_HASHES[name] = digest

unannotated = sorted(set(TRAIN_PATHS) - set(UNIQUE_FILES))
if unannotated:
    warnings.warn(f"{len(unannotated)} train JPEGs have no annotation record; they are not used.")
print("Annotated JPEGs:", len(UNIQUE_FILES), "| annotator records:", len(IMAGES_BY_ID),
      "| instances:", len(RAW["annotations"]))
print("Records per JPEG:", pd.Series([len(v) for v in RECORD_IDS_BY_FILE.values()]).value_counts().sort_index().to_dict())


## Cell 06 — Disk geometry, flattening, image caches, ground-truth index (V4, unchanged)


In [ ]:
CY, CX = (H - 1) / 2.0, (W - 1) / 2.0
DISK_R = CFG.disk_radius_fraction * min(H, W) / 2.0
_YY, _XX = np.mgrid[0:H, 0:W]
RADIAL = (np.hypot(_YY - CY, _XX - CX) / DISK_R).astype(np.float32)
YY_FLAT = _YY.ravel().astype(np.float64)
XX_FLAT = _XX.ravel().astype(np.float64)
del _YY, _XX
DISK = RADIAL < 1.0
RBIN = np.minimum((RADIAL * CFG.radial_bins).astype(np.int32), CFG.radial_bins - 1)
RBIN_DISK = RBIN[DISK]

IMG_MEAN, IMG_STD = 0.5, 0.25
RAD_MEAN, RAD_STD, RAD_CLIP = 0.6, 0.35, 1.25

def flatten_disk(img):
    """Divide by the radial limb-darkening profile, then by a large local median; off-disk = neutral (1.0 -> 128)."""
    x = img.astype(np.float32)
    nb = CFG.radial_bins
    s = np.bincount(RBIN_DISK, weights=x[DISK], minlength=nb)
    c = np.bincount(RBIN_DISK, minlength=nb)
    valid = c > 0
    prof = np.interp(np.arange(nb), np.flatnonzero(valid), (s[valid] / c[valid]))
    k = 7
    prof = np.convolve(np.pad(prof, k // 2, mode="edge"), np.ones(k) / k, mode="valid")
    ratio = x / np.maximum(prof[RBIN], 1.0).astype(np.float32)
    ratio[~DISK] = 1.0
    small = cv2.resize(ratio, (W // 4, H // 4), interpolation=cv2.INTER_AREA)
    small_u8 = np.clip(small * 128.0, 0, 255).astype(np.uint8)
    ksize = max(3, int(round(CFG.flat_median_kernel * H / 2048)) | 1)
    med = cv2.medianBlur(small_u8, ksize).astype(np.float32) / 128.0
    med = cv2.resize(med, (W, H), interpolation=cv2.INTER_LINEAR)
    flat = ratio / np.maximum(med, 0.25)
    flat[~DISK] = 1.0
    return np.clip(flat * 128.0, 0, 255).astype(np.uint8)

CACHE = LOCAL_ROOT / "cache" / f"v3_{ANN_SHA[:12]}_{H}x{W}_r{CFG.disk_radius_fraction:.4f}_k{CFG.flat_median_kernel}_c{CFG.contact_radius}"
for sub in ("img", "flat", "soft", "contact", "gt"):
    (CACHE / sub).mkdir(parents=True, exist_ok=True)

def cache_key(split, name):
    return f"{split}__{Path(name).stem}"

def cache_image(split, name, path):
    key = cache_key(split, name)
    ip, fp = CACHE / "img" / f"{key}.npy", CACHE / "flat" / f"{key}.npy"
    if ip.is_file() and fp.is_file():
        return
    img = read_gray(path)
    if img.shape != (H, W):
        raise ValueError(f"Wrong JPEG shape {img.shape}: {path}")
    save_npy_atomic(ip, img)
    save_npy_atomic(fp, flatten_disk(img))
def build_gt(name):
    """Flat pixel indices and areas of every annotated filament, per annotator set."""
    gp = CACHE / "gt" / f"{cache_key('train', name)}_v4.pkl"
    if gp.is_file():
        with open(gp, "rb") as f:
            return pickle.load(f)
    records = []
    for rid in RECORD_IDS_BY_FILE[name]:
        idx_list, areas = [], []
        for ann in ANNOTATIONS_BY_ID[rid]:
            m = decode_mask(segmentation_to_rle(ann["segmentation"], H, W))
            if not m.any():
                raise ValueError(f"Empty filament annotation {ann['id']}")
            flat_idx = np.flatnonzero(m).astype(np.int32)
            idx_list.append(flat_idx); areas.append(len(flat_idx))
        records.append({"id": rid, "idx": idx_list, "area": np.asarray(areas, np.float64)})
    gt = {"name": name, "records": records}
    save_pickle(gp, gt)
    return gt

with ThreadPoolExecutor(max_workers=min(8, CPU_COUNT)) as pool:
    list(tqdm(pool.map(lambda n: cache_image("train", n, TRAIN_PATHS[n]), UNIQUE_FILES),
              total=len(UNIQUE_FILES), desc="Cache train images + flattened"))
    list(tqdm(pool.map(lambda n: cache_image("test", n, TEST_PATHS[n]), sorted(TEST_PATHS)),
              total=len(TEST_PATHS), desc="Cache test images + flattened"))
    _gt_list = list(tqdm(pool.map(build_gt, UNIQUE_FILES), total=len(UNIQUE_FILES), desc="Ground-truth index"))

GT_INDEX, N_EMPTY_RECORDS = {}, 0
for g in _gt_list:
    scored = [r for r in g["records"] if len(r["idx"])] if CFG.skip_empty_records else list(g["records"])
    N_EMPTY_RECORDS += len(g["records"]) - len(scored)
    GT_INDEX[g["name"]] = {"name": g["name"], "records": scored, "n_all": len(g["records"])}
del _gt_list

@lru_cache(maxsize=64)
def cached_arrays(key, with_targets=True):
    """(image, flattened image) as memory maps. The second argument is kept for compatibility with V3's functions."""
    return tuple(np.load(CACHE / s / f"{key}.npy", mmap_mode="r") for s in ("img", "flat"))

print("Scored annotator sets:", sum(len(g["records"]) for g in GT_INDEX.values()),
      "| filaments:", sum(len(r["idx"]) for g in GT_INDEX.values() for r in g["records"]),
      "| annotator sets without filaments (skipped: %s): %d" % (CFG.skip_empty_records, N_EMPTY_RECORDS))


## Cell 07 — V3 folds and recipe; out-of-fold and test maps averaged over the map runs


In [ ]:
FOLDS = pd.read_csv(V3_RUN / "folds_v3.csv", dtype={"file_name": str, "group": str, "v1_split": str})
if set(FOLDS.file_name) != set(UNIQUE_FILES):
    raise ValueError("V3's folds_v3.csv does not match this dataset.")
FOLDS["fold"] = FOLDS["fold"].astype(int)
FOLD_OF = dict(zip(FOLDS.file_name, FOLDS.fold))

V3_FROZEN = json.loads((V3_RUN / "frozen_recipe_v3.json").read_text())
V3_REC, V3_SHA8 = V3_FROZEN["recipe"], V3_FROZEN["sha256"][:8]
_g = V3_REC["geometry"]
GEOM = (float(_g["t_low"]), None if _g["contact"] is None else float(_g["contact"]), int(_g["offset"]))
V3_TAU = None if V3_REC.get("scorer_tau") is None else float(V3_REC["scorer_tau"])
V3_FOLDS = sorted(int(f) for f in V3_REC["folds"])
OOF_TAG = object_hash({"tta": list(V3CFG["tta_final"]), "tile": V3CFG["tile"], "stride": V3CFG["stride"]})[:10]
MAPS_LOCAL = LOCAL_ROOT / "v3_maps"

def stage_npz(task):
    """Drive .npz (fg, ct) -> two local .npy files that CPU workers can memory-map."""
    src, lf, lc = task
    if not (lf.is_file() and lc.is_file()):
        with np.load(src) as z:
            fg, ct = z["fg"], z["ct"]
        save_npy_atomic(lf, fg); save_npy_atomic(lc, ct)
    return str(lf), str(lc)

def find_oof_dir(fold):
    sha = json.loads((V3_RUN / f"fold{fold}" / "done.json").read_text())["best_sha256"][:10]
    exact = V3_RUN / "oof" / f"fold{fold}_{sha}_{OOF_TAG}"
    if exact.is_dir():
        return exact
    cands = sorted((d for d in (V3_RUN / "oof").glob(f"fold{fold}_*") if d.is_dir()),
                   key=lambda d: len(list(d.glob("*.npz"))))
    if not cands:
        raise FileNotFoundError(f"No out-of-fold maps for fold {fold} under {V3_RUN / 'oof'}. Run V3's Cell 19.")
    return cands[-1]


def stage_avg(task):
    """One or more Drive .npz maps (fg, ct uint8) -> their average as two local .npy files."""
    srcs, lf, lc = task
    if not (lf.is_file() and lc.is_file()):
        fgs, cts = [], []
        for s in srcs:
            with np.load(s) as z:
                fgs.append(z["fg"].astype(np.float32)); cts.append(z["ct"].astype(np.float32))
        fg = np.rint(np.mean(fgs, 0)).astype(np.uint8); ct = np.rint(np.mean(cts, 0)).astype(np.uint8)
        save_npy_atomic(lf, fg); save_npy_atomic(lc, ct)
    return str(lf), str(lc)

def oof_dir_of(run_dir, fold):
    """V3: the directory its own done.json points to; other runs: the fold directory with the most maps."""
    if run_dir == V3_RUN:
        return find_oof_dir(fold)
    cands = sorted((d for d in (run_dir / "oof").glob(f"fold{fold}_*") if d.is_dir()), key=lambda d: len(list(d.glob("*.npz"))))
    if not cands:
        raise FileNotFoundError(f"No out-of-fold maps for fold {fold} under {run_dir / 'oof'}")
    return cands[-1]

def test_map_of(run_dir, stem):
    hits = sorted((run_dir / "test_probs").glob(f"{stem}_{V3_SHA8}.npz" if run_dir == V3_RUN else f"{stem}.npz"))
    if len(hits) != 1:
        raise FileNotFoundError(f"Expected one test map for {stem} under {run_dir / 'test_probs'}, found {len(hits)}")
    return hits[0]

MAPS_LOCAL = LOCAL_ROOT / "maps" / TAG
_tasks, _owners = [], []
for fold in V3_FOLDS:
    dirs = [oof_dir_of(r, fold) for r in MAP_DIRS]
    names = sorted(FOLDS.loc[FOLDS.fold == fold, "file_name"])
    if DEBUG_LIMIT:
        names = names[:DEBUG_LIMIT]
    for name in names:
        stem = Path(name).stem
        srcs = [d / f"{stem}.npz" for d in dirs]
        for s in srcs:
            if not s.is_file():
                raise FileNotFoundError(f"Missing out-of-fold map {s}")
        _tasks.append((srcs, MAPS_LOCAL / f"oof_fold{fold}" / f"{stem}_fg.npy", MAPS_LOCAL / f"oof_fold{fold}" / f"{stem}_ct.npy"))
        _owners.append(name)
with ThreadPoolExecutor(max_workers=8) as pool:
    OOF_PATHS = dict(zip(_owners, tqdm(pool.map(stage_avg, _tasks), total=len(_tasks), desc="Stage out-of-fold maps")))

_tasks, _owners = [], []
for name in sorted(TEST_PATHS)[:DEBUG_LIMIT] if DEBUG_LIMIT else sorted(TEST_PATHS):
    stem = Path(name).stem
    _tasks.append(([test_map_of(r, stem) for r in MAP_DIRS], MAPS_LOCAL / "test" / f"{stem}_fg.npy", MAPS_LOCAL / "test" / f"{stem}_ct.npy"))
    _owners.append(name)
with ThreadPoolExecutor(max_workers=8) as pool:
    TEST_MAPS = dict(zip(_owners, tqdm(pool.map(stage_avg, _tasks), total=len(_tasks), desc="Stage test maps")))

print("V3 geometry:", GEOM, "| V3 keep threshold:", V3_TAU, "| maps averaged over", len(MAP_DIRS), "run(s)")
print("Maps staged:", len(OOF_PATHS), "out-of-fold,", len(TEST_MAPS), "test")


## Cell 08 — Exact PQ engine and instance extraction (V4, unchanged)


In [ ]:
STRUCT8 = ndi.generate_binary_structure(2, 2)
STRUCT4 = ndi.generate_binary_structure(2, 1)
CONN = STRUCT8 if CFG.connectivity == 8 else STRUCT4
FEATURES = ("log_area", "mean_p", "max_p", "frac50", "frac70", "frac90", "mean_contact",
            "elongation", "major_len", "radial", "mean_img", "mean_flat")
F_IDX = {k: i for i, k in enumerate(FEATURES)}

def qt(t):
    return int(round(float(t) * 255))

def match_stats(iou):
    """Strict IoU>0.5, one-to-one max-IoU assignment. Returns (sum_iou, tp, fp, fn, records)."""
    n_gt, n_pred = iou.shape
    if n_gt == 0 or n_pred == 0:
        return np.array([0.0, 0, n_pred, n_gt, 1], np.float64)
    q = iou > 0.5
    if not q.any():
        return np.array([0.0, 0, n_pred, n_gt, 1], np.float64)
    rows, cols = np.flatnonzero(q.any(1)), np.flatnonzero(q.any(0))
    sub = np.where(q[np.ix_(rows, cols)], iou[np.ix_(rows, cols)], 0.0)
    r, c = linear_sum_assignment(sub, maximize=True)
    v = sub[r, c]
    v = v[v > 0]
    tp = len(v)
    return np.array([float(v.sum()), tp, n_pred - tp, n_gt - tp, 1], np.float64)

def match_pairs(iou):
    if iou.size == 0 or not (iou > 0.5).any():
        return []
    sub = np.where(iou > 0.5, iou, 0.0)
    r, c = linear_sum_assignment(sub, maximize=True)
    return [(int(a), int(b), float(sub[a, b])) for a, b in zip(r, c) if sub[a, b] > 0]

def summarize(st):
    s, tp, fp, fn, rec = [float(x) for x in st]
    den = tp + 0.5 * (fp + fn)
    return {"pq": s / den if den else 1.0, "sq": s / tp if tp else 0.0, "rq": tp / den if den else 1.0,
            "precision": tp / (tp + fp) if tp + fp else 0.0, "recall": tp / (tp + fn) if tp + fn else 0.0,
            "tp": int(tp), "fp": int(fp), "fn": int(fn), "records": int(rec)}

def relabel(lab):
    if lab.max() == 0:
        return np.zeros(lab.shape, np.int32), 0
    areas = np.bincount(lab.ravel())
    ids = np.flatnonzero(areas[1:]) + 1
    lut = np.zeros(len(areas), np.int32)
    lut[ids] = np.arange(1, len(ids) + 1, dtype=np.int32)
    return lut[lab], len(ids)

def fill_label_holes(lab):
    fg = lab > 0
    holes = ndi.binary_fill_holes(fg) & ~fg
    if not holes.any():
        return lab
    hl, _ = ndi.label(holes, structure=STRUCT4)
    out = lab.copy()
    for i, sl in enumerate(ndi.find_objects(hl), 1):
        if sl is None:
            continue
        esl = (slice(max(sl[0].start - 1, 0), sl[0].stop + 1), slice(max(sl[1].start - 1, 0), sl[1].stop + 1))
        hole = hl[esl] == i
        ring = ndi.binary_dilation(hole, STRUCT8) & ~hole
        vals = lab[esl][ring]
        vals = vals[vals > 0]
        if vals.size:
            out[esl][hole] = np.bincount(vals).argmax()
    return out

def geometry_labels(fg, ct, geom):
    t_low, contact_t, offset = geom
    low = fg >= qt(t_low)
    if contact_t is not None:
        cores = low & (ct < qt(contact_t))
        lab, n = ndi.label(cores, structure=CONN)
        if n:
            areas = np.bincount(lab.ravel(), minlength=n + 1)
            small = areas < CFG.min_core_area
            small[0] = False
            if small.any():
                lab[small[lab]] = 0
        if lab.any():
            dist, (iy, ix) = ndi.distance_transform_edt(lab == 0, return_distances=True, return_indices=True)
            lab = np.where(low & (dist <= CFG.contact_assign_radius), lab[iy, ix], 0).astype(np.int32)
        else:
            lab = np.zeros((fg.shape), np.int32)
    else:
        lab, _ = ndi.label(low, structure=CONN)
        lab = lab.astype(np.int32)
    if CFG.fill_holes and lab.any():
        lab = fill_label_holes(lab)
    for _ in range(max(offset, 0)):
        lab = np.where(lab == 0, ndi.grey_dilation(lab, footprint=STRUCT4), lab)
    for _ in range(max(-offset, 0)):
        lab = np.where(ndi.grey_erosion(lab, footprint=STRUCT4) == lab, lab, 0)
    return relabel(lab)

def component_features(lab, n, fg, ct, img, flat):
    if n == 0:
        return np.zeros((0, len(FEATURES)), np.float32), np.zeros(0, np.float64)
    L = lab.ravel()
    nz = np.flatnonzero(L)
    Lz = L[nz]
    def s(w):
        return np.bincount(Lz, weights=w, minlength=n + 1)[1:]
    area = np.bincount(Lz, minlength=n + 1)[1:].astype(np.float64)
    a = np.maximum(area, 1.0)
    fgz = np.asarray(fg).ravel()[nz]
    p = fgz.astype(np.float64) / 255.0
    mx = np.zeros(n + 1, np.float64)
    np.maximum.at(mx, Lz, fgz.astype(np.float64))
    yz, xz = YY_FLAT[nz], XX_FLAT[nz]
    my, mxx = s(yz) / a, s(xz) / a
    syy = np.maximum(s(yz * yz) / a - my ** 2, 0)
    sxx = np.maximum(s(xz * xz) / a - mxx ** 2, 0)
    sxy = s(yz * xz) / a - my * mxx
    tr, det = syy + sxx, syy * sxx - sxy ** 2
    disc = np.sqrt(np.maximum(tr ** 2 / 4 - det, 0))
    l1, l2 = tr / 2 + disc, np.maximum(tr / 2 - disc, 1e-3)
    feats = np.stack([
        np.log1p(area), s(p) / a, mx[1:] / 255.0,
        s((fgz >= qt(0.5)).astype(np.float64)) / a, s((fgz >= qt(0.7)).astype(np.float64)) / a,
        s((fgz >= qt(0.9)).astype(np.float64)) / a,
        s(np.asarray(ct).ravel()[nz].astype(np.float64) / 255.0) / a,
        np.sqrt((l1 + 1e-3) / l2), 4.0 * np.sqrt(l1),
        np.hypot(my - CY, mxx - CX) / DISK_R,
        s(np.asarray(img).ravel()[nz].astype(np.float64) / 255.0) / a,
        s(np.asarray(flat).ravel()[nz].astype(np.float64) / 128.0) / a,
    ], 1).astype(np.float32)
    return feats, area

def overlap_matrices(lab, n, area, gt):
    L = lab.ravel()
    out = []
    for rec in gt["records"]:
        inter = np.zeros((len(rec["idx"]), n), np.float64)
        if n:
            for j, idx in enumerate(rec["idx"]):
                inter[j] = np.bincount(L[idx], minlength=n + 1)[1:]
        union = rec["area"][:, None] + area[None, :] - inter
        iou = np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)
        out.append((inter, iou))
    return out

def grid_keep(feats, area, flt):
    th, mc, ma = flt
    if len(area) == 0:
        return np.zeros(0, bool)
    return ((np.rint(feats[:, F_IDX["max_p"]] * 255) >= qt(th)) & (feats[:, F_IDX["mean_p"]] >= mc - 1e-9)
            & (area >= ma))

def prefilter_keep(feats, area):
    if len(area) == 0:
        return np.zeros(0, bool)
    return ((np.rint(feats[:, F_IDX["max_p"]] * 255) >= qt(CFG.scorer_prefilter_max))
            & (area >= CFG.scorer_prefilter_area))

def load_maps(fg, ct):
    if isinstance(fg, (str, Path)):
        fg = np.load(fg, mmap_mode="r")
    if isinstance(ct, (str, Path)):
        ct = np.load(ct, mmap_mode="r")
    return np.asarray(fg), np.asarray(ct)

# ---- process-pool workers (fork: they see GT_INDEX, CACHE, CFG of the parent at fork time) ----
def score_image_grid(task):
    name, fg, ct, geoms, filters = task
    fg, ct = load_maps(fg, ct)
    img, flat = cached_arrays(cache_key("train", name), False)
    gt = GT_INDEX[name]
    out = np.zeros((len(geoms), len(filters), 5), np.float64)
    for gi, g in enumerate(geoms):
        lab, n = geometry_labels(fg, ct, g)
        feats, area = component_features(lab, n, fg, ct, img, flat)
        mats = overlap_matrices(lab, n, area, gt)
        for fi, flt in enumerate(filters):
            keep = grid_keep(feats, area, flt)
            for _, iou in mats:
                out[gi, fi] += match_stats(iou[:, keep])
    return name, out

def scorer_rows(task):
    name, fg, ct, geom = task
    fg, ct = load_maps(fg, ct)
    img, flat = cached_arrays(cache_key("train", name), False)
    lab, n = geometry_labels(fg, ct, geom)
    feats, area = component_features(lab, n, fg, ct, img, flat)
    mats = overlap_matrices(lab, n, area, GT_INDEX[name])
    y = np.zeros(n, np.float64)
    for _, iou in mats:
        if iou.shape[0] and n:
            b = iou.max(0)
            y += np.where(b > 0.5, b, 0.0)
    y /= max(len(mats), 1)
    pre = prefilter_keep(feats, area)
    return name, feats[pre], area[pre], y[pre], [iou[:, pre] for _, iou in mats]

_POOL = None
def get_pool(refresh=False):
    """Fork-based CPU pool. Refresh after defining new globals that workers must see."""
    global _POOL
    if refresh and _POOL is not None:
        _POOL.shutdown(wait=True)
        _POOL = None
    if _POOL is None:
        _POOL = ProcessPoolExecutor(max_workers=CPU_POOL, mp_context=mp.get_context("fork"))
    return _POOL

def stats_table(total, geoms, filters):
    rows = []
    for gi, g in enumerate(geoms):
        for fi, f in enumerate(filters):
            rows.append({"t_low": g[0], "contact": g[1], "offset": g[2], "t_high": f[0], "min_mean": f[1],
                         "min_area": f[2], **summarize(total[gi, fi])})
    return pd.DataFrame(rows)

def pick_best(table):
    t = table.copy()
    t["_c"] = t["contact"].isna()
    t["_o"] = (t["offset"] == 0)
    t = t.sort_values(["pq", "_c", "_o", "min_area", "min_mean"], ascending=[False, False, False, True, True])
    return t.iloc[0].drop(["_c", "_o"]).to_dict()

def as_geom(row):
    c = row["contact"]
    return (float(row["t_low"]), None if (c is None or (isinstance(c, float) and np.isnan(c))) else float(c), int(row["offset"]))

def as_filter(row):
    return (float(row["t_high"]), float(row["min_mean"]), int(row["min_area"]))

print("PQ engine ready. Features:", FEATURES)


## Cell 09 — V4's cached detections (each training image by the detector that never saw it, test by all four)


In [ ]:
DET_FOLDS, DET_OOF, _test_by_fold = [], {}, {}
for fold in range(CFG.n_folds):
    oof_f, test_f = sorted(DET_DIR.glob(f"oof_fold{fold}_*.pkl")), sorted(DET_DIR.glob(f"test_fold{fold}_*.pkl"))
    if len(oof_f) != 1 or len(test_f) != 1:
        raise FileNotFoundError(f"Expected one oof and one test detection file for fold {fold} in {DET_DIR}")
    with open(oof_f[0], "rb") as f:
        DET_OOF.update(pickle.load(f))
    with open(test_f[0], "rb") as f:
        _test_by_fold[fold] = pickle.load(f)
    DET_FOLDS.append(fold)
DET_TEST = {n: [_test_by_fold[f][n] for f in DET_FOLDS] for n in sorted(TEST_MAPS)}
print("Detectors:", DET_FOLDS, "| out-of-fold images with detections:", len(DET_OOF),
      "| mean detections per image:", round(float(np.mean([len(d['scores']) for d in DET_OOF.values()])), 1))


## Cell 10 — Fusion: detector evidence, detector-only candidates (V4) + extra-level training rows (V5 step 1)


In [ ]:
DET_FEATURES = ("det_score", "det_iou", "det_cov")
ALL_FEATURES = tuple(FEATURES) + DET_FEATURES + ("t_low", "src")     # V5: t_low = hysteresis level of the candidate
N_V3F, N_DETF = len(FEATURES), len(DET_FEATURES)
_GROW_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * CFG.det_grow + 1, 2 * CFG.det_grow + 1))

def rasterize_detections(dets):
    """Detections of one detector -> list of (score, y0, x0, boolean mask of its bounding window), best first."""
    out = []
    for s, p in zip(dets["scores"], dets["polys"]):
        if p is None or len(p) < 3:
            continue
        q = np.rint(np.asarray(p, np.float64)).astype(np.int32)
        x0, y0 = max(int(q[:, 0].min()), 0), max(int(q[:, 1].min()), 0)
        x1, y1 = min(int(q[:, 0].max()) + 1, W), min(int(q[:, 1].max()) + 1, H)
        if x1 - x0 < 2 or y1 - y0 < 2:
            continue
        roi = np.zeros((y1 - y0, x1 - x0), np.uint8)
        cv2.fillPoly(roi, [q - np.array([x0, y0], np.int32)], 1)
        if int(roi.sum()) < CFG.det_min_area:
            continue
        out.append((float(s), y0, x0, roi.astype(bool)))
    out.sort(key=lambda t: -t[0])
    return out

def detection_evidence(lab, n, area, rast):
    """Per candidate: score of the detection sharing most pixels with it, IoU with that detection, fraction covered."""
    score, iou, cover = np.zeros(n), np.zeros(n), np.zeros(n)
    if n == 0 or not rast:
        return score, iou, cover
    best = np.zeros(n)
    covered = np.zeros(lab.shape, bool)
    for s, y0, x0, m in rast:
        h, w = m.shape
        covered[y0:y0 + h, x0:x0 + w] |= m
        inter = np.bincount(lab[y0:y0 + h, x0:x0 + w][m], minlength=n + 1)[1:].astype(np.float64)
        a = float(m.sum())
        upd = (inter >= CFG.det_link_frac * np.minimum(area, a)) & (inter > best)
        if upd.any():
            best[upd] = inter[upd]
            score[upd] = s
            iou[upd] = inter[upd] / (area[upd] + a - inter[upd])
    cover = np.bincount(lab[covered], minlength=n + 1)[1:] / np.maximum(area, 1.0)
    return score, iou, cover

def largest_piece(mask):
    lab, n = ndi.label(mask, structure=CONN)
    if n <= 1:
        return mask
    sizes = np.bincount(lab.ravel())
    sizes[0] = 0
    return lab == int(sizes.argmax())

def consensus_scores(rasts):
    """Every detection gets the mean, over all detectors, of the best score among that detector's detections
    that overlap it (IoU >= 0.3). With one detector this is its own score; with four, a detection found by only
    one of them is worth a quarter. Keeps test images (4 detectors) comparable to out-of-fold images (1 detector)."""
    items = [(d, s, y0, x0, m, float(m.sum())) for d, rast in enumerate(rasts) for s, y0, x0, m in rast]
    out = []
    for d, s, y0, x0, m, a in items:
        total = s
        for d2 in range(len(rasts)):
            if d2 == d:
                continue
            best = 0.0
            for s2, v0, u0, m2 in rasts[d2]:
                if s2 <= best:
                    break                                         # sorted by score: nothing better follows
                ya, yb = max(y0, v0), min(y0 + m.shape[0], v0 + m2.shape[0])
                xa, xb = max(x0, u0), min(x0 + m.shape[1], u0 + m2.shape[1])
                if ya >= yb or xa >= xb:
                    continue
                inter = float((m[ya - y0:yb - y0, xa - x0:xb - x0] & m2[ya - v0:yb - v0, xa - u0:xb - u0]).sum())
                if inter >= 0.3 * (a + float(m2.sum()) - inter):
                    best = s2
            total += best
        out.append((total / max(len(rasts), 1), y0, x0, m))
    out.sort(key=lambda t: -t[0])
    return out

def detector_only_candidates(lab, fg, rasts):
    """Objects the detectors found where the UNet has no candidate. The mask is the UNet's probability inside the
    (slightly grown) detection when there is enough of it, otherwise the detection itself. All disjoint."""
    pooled = consensus_scores(rasts)
    occupied = lab > 0
    new = np.zeros(lab.shape, np.int32)
    kinds, g = [], CFG.det_grow
    for s, y0, x0, m in pooled:
        if s < CFG.det_only_floor:
            break
        h, w = m.shape
        if int((m & occupied[y0:y0 + h, x0:x0 + w]).sum()) > CFG.det_only_max_claimed * int(m.sum()):
            continue                                              # already represented
        ya, xa, yb, xb = max(y0 - g, 0), max(x0 - g, 0), min(y0 + h + g, H), min(x0 + w + g, W)
        big = np.zeros((yb - ya, xb - xa), np.uint8)
        big[y0 - ya:y0 - ya + h, x0 - xa:x0 - xa + w] = m
        free = ~occupied[ya:yb, xa:xb]
        det_free = (big > 0) & free
        claim = (cv2.dilate(big, _GROW_KERNEL) > 0) & free & (fg[ya:yb, xa:xb] >= qt(CFG.det_claim_t))
        if int(claim.sum()) >= CFG.det_claim_min_frac * int(det_free.sum()):
            mask, kind = largest_piece(claim), 1                  # drawn by the UNet
        else:
            mask, kind = largest_piece(det_free), 2               # drawn by the detector
        if int(mask.sum()) < CFG.det_only_min_area:
            continue
        kinds.append(kind)
        new[ya:yb, xa:xb][mask] = len(kinds)
        occupied[ya:yb, xa:xb] |= mask
    return new, np.asarray(kinds, np.float32)

def build_candidates(fg, ct, img, flat, det_list, with_detector_only=True):
    """V3 candidates (same geometry and prefilter as V3) + detector-only candidates, with all features.
    Returns (label map, n candidates, n from V3, feature matrix [n, len(ALL_FEATURES)], areas)."""
    lab0, n0 = geometry_labels(fg, ct, GEOM)
    feats, area = component_features(lab0, n0, fg, ct, img, flat)
    pre = prefilter_keep(feats, area)
    lut = np.zeros(n0 + 1, np.int32)
    ids = np.flatnonzero(pre) + 1
    lut[ids] = np.arange(1, len(ids) + 1, dtype=np.int32)
    lab, n_v3 = lut[lab0], len(ids)
    feats, area = feats[pre], area[pre]
    rasts = [rasterize_detections(d) for d in det_list]
    kinds = np.zeros(0, np.float32)
    if with_detector_only and any(rasts):
        new, kinds = detector_only_candidates(lab, fg, rasts)
        if len(kinds):
            lab = np.where(new > 0, new + n_v3, lab)
            feats, area = component_features(lab, n_v3 + len(kinds), fg, ct, img, flat)
    n = n_v3 + len(kinds)
    ev = np.zeros((n, N_DETF))
    for rast in rasts:
        ev += np.stack(detection_evidence(lab, n, area, rast), 1)
    if rasts:
        ev /= len(rasts)
    src = np.r_[np.zeros(n_v3, np.float32), kinds]
    lev = np.full((n, 1), GEOM[0], np.float32)
    X = np.concatenate([feats.astype(np.float32), ev.astype(np.float32), lev, src[:, None].astype(np.float32)], 1)
    return lab, n, n_v3, X, area

def level_rows(fg, ct, img, flat, det_list, gt):
    """V5 step 1: candidates at the extra hysteresis levels (same contact / offset / prefilter, no detector-only),
    with their features and labels. Used only as additional training rows for the keep-scorer."""
    rasts = [rasterize_detections(d) for d in det_list]
    Xs, ys = [], []
    for t in CFG.aug_levels:
        lab0, n0 = geometry_labels(fg, ct, (float(t), GEOM[1], GEOM[2]))
        feats, area = component_features(lab0, n0, fg, ct, img, flat)
        pre = prefilter_keep(feats, area)
        if not pre.any():
            continue
        lut = np.zeros(n0 + 1, np.int32)
        ids = np.flatnonzero(pre) + 1
        lut[ids] = np.arange(1, len(ids) + 1, dtype=np.int32)
        lab, n = lut[lab0], len(ids)
        feats, area = feats[pre], area[pre]
        ev = np.zeros((n, N_DETF))
        for rast in rasts:
            ev += np.stack(detection_evidence(lab, n, area, rast), 1)
        if rasts:
            ev /= len(rasts)
        mats = overlap_matrices(lab, n, area, gt)
        y = np.zeros(n)
        for _, iou in mats:
            if iou.shape[0]:
                b = iou.max(0)
                y += np.where(b > 0.5, b, 0.0)
        y /= max(len(mats), 1)
        Xs.append(np.concatenate([feats.astype(np.float32), ev.astype(np.float32), np.full((n, 1), t, np.float32),
                                  np.zeros((n, 1), np.float32)], 1))
        ys.append(y)
    if not Xs:
        return np.zeros((0, len(ALL_FEATURES)), np.float32), np.zeros(0)
    return np.concatenate(Xs), np.concatenate(ys)

def fusion_tests():
    global GEOM
    saved, GEOM = GEOM, (0.4, None, 0)                             # fixed geometry for the test only
    try:
        _fusion_tests()
    finally:
        GEOM = saved

def _fusion_tests():
    fg = np.zeros((H, W), np.uint8); ct = np.zeros((H, W), np.uint8)
    img = np.full((H, W), 120, np.uint8); flat = np.full((H, W), 128, np.uint8)
    fg[100:110, 100:300] = 230                                     # A: UNet candidate, also detected
    fg[200:210, 100:300] = 220                                     # B: UNet candidate, not detected
    fg[300:308, 100:260] = 70                                      # C: below the UNet threshold, detected -> UNet-drawn
    box = lambda y0, y1, x0, x1: np.array([[x0, y0], [x1, y0], [x1, y1], [x0, y1]], np.float32)
    dets = {"scores": np.array([0.9, 0.6, 0.4, 0.03], np.float32),
            "polys": [box(99, 110, 98, 301), box(299, 308, 98, 262), box(400, 409, 100, 250), box(450, 459, 100, 250)]}
    lab, n, n_v3, X, area = build_candidates(fg, ct, img, flat, [dets])
    assert n_v3 == 2 and n == 4, (n_v3, n)                         # A, B + C (UNet-drawn) + D (detector-drawn); 0.03 ignored
    col = {k: i for i, k in enumerate(ALL_FEATURES)}
    a, b = lab[105, 200] - 1, lab[205, 200] - 1
    assert X[a, col["det_score"]] > 0.85 and X[a, col["det_iou"]] > 0.7 and X[a, col["det_cov"]] > 0.95
    assert X[b, col["det_score"]] == 0 and X[b, col["det_cov"]] == 0
    c, d = lab[304, 150] - 1, lab[404, 150] - 1
    assert X[c, col["src"]] == 1 and X[d, col["src"]] == 2 and lab[454, 150] == 0
    assert abs(X[c, col["det_score"]] - 0.6) < 1e-6 and abs(X[d, col["det_score"]] - 0.4) < 1e-6
    lab2, n2, _, X2, _ = build_candidates(fg, ct, img, flat, [dets, {"scores": np.zeros(0, np.float32), "polys": []}])
    assert n2 == 4 and abs(X2[a, col["det_score"]] - X[a, col["det_score"]] / 2) < 1e-6     # mean over detectors
    lab3, n3, n3v, X3, _ = build_candidates(fg, ct, img, flat, [dets], with_detector_only=False)
    assert n3 == n3v == 2 and np.allclose(X3[:, :N_V3F + N_DETF], X[:2, :N_V3F + N_DETF])
    weak = {"scores": np.array([0.15], np.float32), "polys": [box(400, 409, 100, 250)]}
    none = {"scores": np.zeros(0, np.float32), "polys": []}
    assert build_candidates(fg, ct, img, flat, [weak])[1] == 3                 # one detector at 0.15: candidate
    assert build_candidates(fg, ct, img, flat, [weak, none])[1] == 2           # found by 1 of 2: 0.075 < floor, none
    assert build_candidates(fg, ct, img, flat, [weak, weak])[1] == 3           # found by both: candidate again
    for k in range(1, n + 1):                                      # every candidate is one non-empty, disjoint region
        assert (lab == k).any()
    print("PASS: detector evidence, detector-only candidates (UNet-drawn and detector-drawn), consensus over detectors")

fusion_tests()


## Cell 11 — Fusion candidates for every out-of-fold image (+ extra-level rows when `SFSC_AUG=1`)


In [ ]:
EVAL_NAMES = sorted(n for n in OOF_PATHS if n in DET_OOF)
if not EVAL_NAMES:
    raise RuntimeError("No image has both a V3 out-of-fold map and detector predictions.")

def oof_task(task):
    name, fgp, ctp, dets = task
    fg, ct = load_maps(fgp, ctp)
    img, flat = cached_arrays(cache_key("train", name), False)
    lab, n, n_v3, X, area = build_candidates(fg, ct, img, flat, [dets])
    mats = overlap_matrices(lab, n, area, GT_INDEX[name])
    y = np.zeros(n)
    for _, iou in mats:
        if iou.shape[0] and n:
            b = iou.max(0)
            y += np.where(b > 0.5, b, 0.0)
    y /= max(len(mats), 1)
    ax, ay = level_rows(fg, ct, img, flat, [dets], GT_INDEX[name]) if AUG else (None, None)
    return name, X, area, y, len(mats), [iou for _, iou in mats], ax, ay

pool = get_pool(refresh=True)                     # workers must see the fusion functions and the ground-truth index
_rows = list(tqdm(pool.map(oof_task, [(n, *OOF_PATHS[n], DET_OOF[n]) for n in EVAL_NAMES], chunksize=1),
                  total=len(EVAL_NAMES), desc="Fusion candidates (out-of-fold)"))
assert [r[0] for r in _rows] == EVAL_NAMES

CAND_X = np.concatenate([r[1] for r in _rows]) if _rows else np.zeros((0, len(ALL_FEATURES)), np.float32)
CAND_Y = np.concatenate([r[3] for r in _rows])
_counts = [len(r[3]) for r in _rows]
CAND_OFF = np.r_[0, np.cumsum(_counts)]                                  # candidates of image i: OFF[i]:OFF[i+1]
CAND_IMG = np.repeat(np.arange(len(_rows)), _counts)
CAND_K = np.repeat(np.array([r[4] for r in _rows], np.float64), _counts)  # annotator sets scoring each candidate
CAND_SRC = CAND_X[:, -1]
IOUS = [r[5] for r in _rows]
if AUG:
    AUG_X = np.concatenate([r[6] for r in _rows])
    AUG_Y = np.concatenate([r[7] for r in _rows])
    _ac = [len(r[7]) for r in _rows]
    AUG_IMG = np.repeat(np.arange(len(_rows)), _ac)
    AUG_K = np.repeat(np.array([r[4] for r in _rows], np.float64), _ac)
    print("Extra-level training rows:", len(AUG_Y), f"({len(AUG_Y) / len(_rows):.1f} per image,",
          f"{100 * (AUG_Y > 0).mean():.1f}% match at least one annotator)")                                             # per image: one [n_gt, n_cand] IoU matrix per annotator set
IMG_FOLD = np.array([FOLD_OF[n] for n in EVAL_NAMES])
IMG_NREC = np.array([GT_INDEX[n]["n_all"] for n in EVAL_NAMES])
del _rows
print("Images:", len(EVAL_NAMES), "| candidates:", len(CAND_Y), "| from the UNet:", int((CAND_SRC == 0).sum()),
      "| detector-only, UNet-drawn:", int((CAND_SRC == 1).sum()), "| detector-only, detector-drawn:", int((CAND_SRC == 2).sum()))
_hit = CAND_Y > 0
for _label, _m in (("UNet candidates", CAND_SRC == 0), ("detector-only candidates", CAND_SRC > 0)):
    if _m.any():
        print(f"  {_label}: {100 * _hit[_m].mean():.1f}% match at least one annotator")


## Cell 12 — Cross-fitted scorers, threshold choice, paired bootstrap, selection (V4 + `v5_aug`)


In [ ]:
def make_hgb():
    return HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, max_leaf_nodes=31,
                                         min_samples_leaf=40, l2_regularization=1.0, random_state=CFG.seed)

_all = np.ones(len(CAND_Y), bool)
VARIANTS = {   # name: (which candidates, feature columns [start, stop), weight by number of annotator sets)
    "v3_replica":  (CAND_SRC == 0, (0, N_V3F), False),               # V3's scorer exactly as V3 fitted it
    "v3_rescored": (CAND_SRC == 0, (0, N_V3F), True),                # baseline every V4 variant must beat
    "v4_evidence": (CAND_SRC == 0, (0, N_V3F + N_DETF), True),       # V3 candidates + detector evidence
    "v4_full":     (_all, (0, len(ALL_FEATURES)), True),             # ... + detector-only candidates
}
if AUG:
    VARIANTS["v5_aug"] = (_all, (0, len(ALL_FEATURES)), True)         # v4_full + extra-level training rows
BASELINE = "v3_rescored"

_folds_present = np.unique(IMG_FOLD)
if len(_folds_present) >= 2:
    IMG_GROUP = IMG_FOLD.copy()
else:                                                              # debug run with one fold: 3 pseudo-folds by file name
    IMG_GROUP = np.array([int(hashlib.sha256(n.encode()).hexdigest()[:6], 16) % 3 for n in EVAL_NAMES])
CAND_GROUP = IMG_GROUP[CAND_IMG]
AUG_GROUP = IMG_GROUP[AUG_IMG] if AUG else None

def crossfit(sel, cols, weighted, aug=False):
    """Each candidate is scored by a model that never saw its image's fold. Unselected candidates get -inf.
    aug=True adds the extra-level rows of the training folds to every fit (they are never scored)."""
    u = np.full(len(CAND_Y), -np.inf)
    X = CAND_X[:, cols[0]:cols[1]]
    for g in np.unique(CAND_GROUP):
        tr, te = sel & (CAND_GROUP != g), sel & (CAND_GROUP == g)
        if tr.sum() < (40 if CFG.debug_run else 200) or te.sum() == 0:
            if te.any():
                u[te] = 0.0
            continue
        Xtr, ytr, ktr = X[tr], CAND_Y[tr], CAND_K[tr]
        if aug:
            at = AUG_GROUP != g
            Xtr = np.concatenate([Xtr, AUG_X[at][:, cols[0]:cols[1]]])
            ytr, ktr = np.r_[ytr, AUG_Y[at]], np.r_[ktr, AUG_K[at]]
        model = make_hgb().fit(Xtr, ytr, sample_weight=ktr if weighted else None)
        u[te] = model.predict(X[te])
    return u

def image_stats(u, tau):
    """Per image: [sum IoU, TP, FP, FN, annotator sets], pooled over that image's annotator sets."""
    out = np.zeros((len(EVAL_NAMES), 5))
    for i in range(len(EVAL_NAMES)):
        keep = u[CAND_OFF[i]:CAND_OFF[i + 1]] >= tau
        for iou in IOUS[i]:
            out[i] += match_stats(iou[:, keep])
    return out

def pq_value(st):
    den = st[1] + 0.5 * (st[2] + st[3])
    return float(st[0] / den) if den else 0.0

def paired_bootstrap(a, b, n=None):
    """95% interval of PQ(a) - PQ(b), resampling images (all annotator sets of an image stay together)."""
    n = CFG.bootstrap if n is None else n
    rng = np.random.default_rng(CFG.seed)
    w = rng.multinomial(len(a), np.full(len(a), 1.0 / len(a)), size=n).astype(np.float64)
    sa, sb = w @ a[:, :4], w @ b[:, :4]
    pq = lambda s: s[:, 0] / np.maximum(s[:, 1] + 0.5 * (s[:, 2] + s[:, 3]), 1e-9)
    return np.percentile(pq(sa) - pq(sb), [2.5, 97.5])

RESULT = {}
for vname, (sel, cols, weighted) in VARIANTS.items():
    u = crossfit(sel, cols, weighted, aug=(vname == "v5_aug"))
    rows = [{"tau": float(t), **summarize(image_stats(u, t).sum(0))} for t in CFG.scorer_taus]
    table = pd.DataFrame(rows)
    best = table.sort_values(["pq", "tau"], ascending=[False, True]).iloc[0]
    RESULT[vname] = {"u": u, "table": table, "tau": float(best.tau), "stats": image_stats(u, float(best.tau))}
    table.to_csv(RUN / f"tau_table_{vname}.csv", index=False)

# ---- check: the replica must reproduce V3's own cross-fitted score when scored the way V3 scored it ----
if V3_TAU is not None:
    _rep = summarize(image_stats(RESULT["v3_replica"]["u"], V3_TAU).sum(0))["pq"]
    _ref = V3_REC.get("oof_pq_scorer")
    _same_setup = len(EVAL_NAMES) == len(UNIQUE_FILES) and (N_EMPTY_RECORDS == 0 or not CFG.skip_empty_records)
    print(f"V3 replica at V3's threshold {V3_TAU:g}: PQ {_rep:.4f} | V3 reported {_ref if _ref is None else round(_ref, 4)}"
          + ("" if _same_setup else "  (not comparable: fewer images, or annotator sets without filaments are skipped here)"))

base_stats = RESULT[BASELINE]["stats"]
folds_here = sorted(np.unique(IMG_GROUP))
SUMMARY = []
for vname, r in RESULT.items():
    s = summarize(r["stats"].sum(0))
    fold_pq = {g: pq_value(r["stats"][IMG_GROUP == g].sum(0)) for g in folds_here}
    base_fold = {g: pq_value(base_stats[IMG_GROUP == g].sum(0)) for g in folds_here}
    lo, hi = paired_bootstrap(r["stats"], base_stats) if vname != BASELINE else (0.0, 0.0)
    SUMMARY.append({"variant": vname, "tau": r["tau"], "pq": s["pq"], "sq": s["sq"], "rq": s["rq"],
                    "precision": s["precision"], "recall": s["recall"], "tp": s["tp"], "fp": s["fp"], "fn": s["fn"],
                    "gain": s["pq"] - summarize(base_stats.sum(0))["pq"], "ci_low": float(lo), "ci_high": float(hi),
                    "fold_wins": int(sum(fold_pq[g] > base_fold[g] for g in folds_here)), "folds": len(folds_here)})
SUMMARY = pd.DataFrame(SUMMARY)
SUMMARY.to_csv(RUN / "selection_v5.csv", index=False)
print("\nCross-fitted out-of-fold results (gain, interval and fold wins are against", BASELINE + "):")
print(SUMMARY.round(4).to_string(index=False))

need_wins = min(CFG.min_fold_wins, len(folds_here))
eligible = SUMMARY[SUMMARY.variant.isin(["v4_evidence", "v4_full"]) & (SUMMARY.gain >= CFG.min_gain)
                   & (SUMMARY.fold_wins >= need_wins)]
ADOPTED = eligible.sort_values("pq", ascending=False).iloc[0].variant if len(eligible) else BASELINE
if AUG:                                     # V5 step 1 must beat v4_full itself, not re-scored V3
    _a, _b = RESULT["v5_aug"]["stats"], RESULT["v4_full"]["stats"]
    _gain = pq_value(_a.sum(0)) - pq_value(_b.sum(0))
    _wins = int(sum(pq_value(_a[IMG_GROUP == g].sum(0)) > pq_value(_b[IMG_GROUP == g].sum(0)) for g in folds_here))
    _lo, _hi = paired_bootstrap(_a, _b)
    AUG_VS_V4 = {"gain": _gain, "fold_wins": _wins, "ci_low": float(_lo), "ci_high": float(_hi)}
    print(f"\nv5_aug vs v4_full: {_gain:+.4f} PQ, 95% interval {_lo:+.4f} to {_hi:+.4f}, wins {_wins}/{len(folds_here)} folds")
    if _gain >= CFG.min_gain and _wins >= need_wins:
        ADOPTED = "v5_aug"
np.savez_compressed(RUN / "image_stats.npz", names=np.array(EVAL_NAMES), folds=IMG_GROUP, off=CAND_OFF, src=CAND_SRC,
                    **{f"{v}__tau{r['tau']:g}": r["stats"] for v, r in RESULT.items()},
                    **{f"{v}__u": r["u"] for v, r in RESULT.items()})
print(f"\nAdopted: {ADOPTED}"
      + ("" if ADOPTED != BASELINE else f"  (no V4 variant gained {CFG.min_gain} PQ with wins in {need_wins} folds)"))

# ---- final scorers: fitted on every out-of-fold candidate, used only on test images ----
FINAL = {}
for vname in ("v3_rescored", "v4_evidence", "v4_full") + (("v5_aug",) if AUG else ()):
    sel, cols, weighted = VARIANTS[vname]
    Xf, yf, kf = CAND_X[sel][:, cols[0]:cols[1]], CAND_Y[sel], CAND_K[sel]
    if vname == "v5_aug":
        Xf, yf, kf = np.concatenate([Xf, AUG_X[:, cols[0]:cols[1]]]), np.r_[yf, AUG_Y], np.r_[kf, AUG_K]
    model = make_hgb().fit(Xf, yf, sample_weight=kf if weighted else None)
    FINAL[vname] = {"model": model, "tau": RESULT[vname]["tau"], "cols": cols, "all": vname in ("v4_full", "v5_aug")}
save_pickle(RUN / "scorer_v5.pkl", {"final": FINAL, "adopted": ADOPTED, "features": ALL_FEATURES, "geometry": GEOM})
write_json(RUN / "selection_v5.json", {"maps": MAP_RUNS, "aug": AUG, "aug_vs_v4_full": AUG_VS_V4 if AUG else None, "adopted": ADOPTED, "baseline": BASELINE, "min_gain": CFG.min_gain,
                                       "summary": SUMMARY.to_dict("records"), "detector_folds": DET_FOLDS,
                                       "images": len(EVAL_NAMES), "skip_empty_records": CFG.skip_empty_records})


## Cell 13 — Diagnostics for the adopted variant (only with `SFSC_DIAG=1`)


In [ ]:
if CFG.run_diagnostics:
    SIZE_EDGES = (500, 1500, 5000)
    def size_bucket(a):
        return "tiny(<500)" if a < SIZE_EDGES[0] else "small" if a < SIZE_EDGES[1] else "medium" if a < SIZE_EDGES[2] else "large(>=5000)"
    def radial_bucket(r):
        return "centre(<0.6)" if r < 0.6 else ("mid(0.6-0.85)" if r < 0.85 else "limb(>=0.85)")

    def census_task(task):
        """Final out-of-fold prediction of one image (cross-fitted keep decisions) compared with every annotator set."""
        name, fgp, ctp, dets, keep = task
        fg, ct = load_maps(fgp, ctp)
        img, flat = cached_arrays(cache_key("train", name), False)
        lab, n, n_v3, X, area = build_candidates(fg, ct, img, flat, [dets])
        if n != len(keep):
            raise RuntimeError(f"{name}: candidates changed between cells ({n} vs {len(keep)})")
        lut = np.zeros(n + 1, np.int32)
        ids = np.flatnonzero(keep) + 1
        lut[ids] = np.arange(1, len(ids) + 1, dtype=np.int32)
        lab, n, area, src = lut[lab], len(ids), area[keep], X[keep, -1]
        L = lab.ravel()
        recs = GT_INDEX[name]["records"]
        mats = overlap_matrices(lab, n, area, GT_INDEX[name])
        rec_labels = []
        if len(recs) > 1:                                  # per annotator set: which filament owns each pixel
            for rec in recs:
                v = np.zeros(H * W, np.int32)
                for k, idx in enumerate(rec["idx"]):
                    v[idx] = k + 1
                rec_labels.append(v)
        fgv = np.asarray(fg).ravel()
        rast = rasterize_detections(dets)

        def detector_iou(idx, a_gt):
            """Best IoU between this annotated filament and any raw detector mask."""
            ys, xs = idx // W, idx % W
            best = 0.0
            for _, y0, x0, m in rast:
                inside = (ys >= y0) & (ys < y0 + m.shape[0]) & (xs >= x0) & (xs < x0 + m.shape[1])
                if inside.any():
                    i = float(m[ys[inside] - y0, xs[inside] - x0].sum())
                    best = max(best, i / (a_gt + float(m.sum()) - i))
            return best
        gt_rows, pred_rows, pair_iou, pair_dice = [], [], [], []
        gt_deg, pred_deg = Counter(), Counter()
        for ri, ((inter, iou), rec) in enumerate(zip(mats, recs)):
            pairs = match_pairs(iou)
            mg, mp_ = {a for a, _, _ in pairs}, {b for _, b, _ in pairs}
            if n and len(rec["idx"]):
                hit = inter > 0
                pair_iou.append(iou[hit]); pair_dice.append((2 * inter / (rec["area"][:, None] + area[None, :]))[hit])
                gt_deg.update(np.minimum(hit.sum(1), 2).tolist()); pred_deg.update(np.minimum(hit.sum(0), 2).tolist())
            else:
                gt_deg.update([0] * len(rec["idx"])); pred_deg.update([0] * n)
            for j, idx in enumerate(rec["idx"]):
                a_gt = rec["area"][j]
                rad = math.hypot(YY_FLAT[idx].mean() - CY, XX_FLAT[idx].mean() - CX) / DISK_R
                best = float(iou[j].max()) if n else 0.0
                tail_p = np.nan
                if j in mg:
                    cat = "matched"
                else:
                    cover = inter[j] / a_gt if n else np.zeros(0)
                    c_best = int(np.argmax(inter[j])) if n and inter[j].max() > 0 else -1
                    merged = c_best >= 0 and any(k != j and inter[k, c_best] >= 0.2 * rec["area"][k] for k in range(len(rec["idx"])))
                    if n and (cover >= 0.2).sum() >= 2:
                        cat = "fragmented"
                    elif merged:
                        cat = "merged"
                    elif best >= 0.1:
                        cat = "near_miss(0.1-0.5)"
                        left_out = idx[L[idx] != int(np.argmax(iou[j])) + 1]
                        if len(left_out):
                            tail_p = float(np.median(fgv[left_out])) / 255.0
                    else:
                        cat = "missed(<0.1)"
                support = np.nan
                if len(recs) > 1:                          # how many OTHER annotators drew this filament (IoU > 0.5)
                    support = 0
                    for r2, other in enumerate(recs):
                        if r2 == ri or not len(other["idx"]):
                            continue
                        cnt = np.bincount(rec_labels[r2][idx], minlength=len(other["idx"]) + 1)[1:].astype(np.float64)
                        support += int((cnt / (a_gt + other["area"] - cnt)).max() > 0.5)
                gt_rows.append({"file_name": name, "record": ri, "area": a_gt, "size": size_bucket(a_gt),
                                "radial": radial_bucket(rad), "best_iou": best, "category": cat,
                                "n_records": len(recs), "others_who_drew_it": support, "tail_prob": tail_p,
                                "gt_mean_prob": float(fgv[idx].mean()) / 255.0,
                                "detector_best_iou": np.nan if j in mg else detector_iou(idx, a_gt)})
            for c in range(n):
                best = float(iou[:, c].max()) if iou.shape[0] else 0.0
                cat = "tp" if c in mp_ else ("near_miss_fp" if best >= 0.1 else "spurious_fp")
                elsewhere = any(r2 != ri and m2[1].shape[0] and m2[1][:, c].max() > 0.5 for r2, m2 in enumerate(mats))
                pred_rows.append({"file_name": name, "record": ri, "area": area[c], "size": size_bucket(area[c]),
                                  "source": ("unet", "detector+unet pixels", "detector mask")[int(src[c])],
                                  "best_iou": best, "category": cat, "n_records": len(recs),
                                  "matches_another_annotator": bool(elsewhere)})
        cat_iou = np.concatenate(pair_iou) if pair_iou else np.zeros(0)
        cat_dice = np.concatenate(pair_dice) if pair_dice else np.zeros(0)
        return gt_rows, pred_rows, cat_iou, cat_dice, gt_deg, pred_deg

    _u, _tau = RESULT[ADOPTED]["u"], RESULT[ADOPTED]["tau"]
    pool = get_pool(refresh=True)
    _tasks = [(n, *OOF_PATHS[n], DET_OOF[n], _u[CAND_OFF[i]:CAND_OFF[i + 1]] >= _tau) for i, n in enumerate(EVAL_NAMES)]
    GT_ROWS, PRED_ROWS, _pi, _pd, GT_DEG, PRED_DEG = [], [], [], [], Counter(), Counter()
    for g, p, a, b, gd, pdg in tqdm(pool.map(census_task, _tasks, chunksize=1), total=len(_tasks), desc=f"Error census ({ADOPTED})"):
        GT_ROWS += g; PRED_ROWS += p; _pi.append(a); _pd.append(b); GT_DEG.update(gd); PRED_DEG.update(pdg)
    GT_DF, PRED_DF = pd.DataFrame(GT_ROWS), pd.DataFrame(PRED_ROWS)
    DIAG = RUN / "diagnostics"; DIAG.mkdir(exist_ok=True)
    GT_DF.to_csv(DIAG / "oof_gt_instances.csv", index=False); PRED_DF.to_csv(DIAG / "oof_predictions.csv", index=False)
    PAIR_IOU, PAIR_DICE = np.concatenate(_pi) if _pi else np.zeros(0), np.concatenate(_pd) if _pd else np.zeros(0)
    pct = lambda t: (t * 100).round(1).to_string()

    print(f"Adopted variant: {ADOPTED} at threshold {_tau:g} (cross-fitted, out-of-fold)\n")
    print("PQ by number of annotators per image:")
    _st = RESULT[ADOPTED]["stats"]
    for k in sorted(np.unique(IMG_NREC)):
        s = summarize(_st[IMG_NREC == k].sum(0))
        print(f"  {k} annotator(s): PQ {s['pq']:.4f}  SQ {s['sq']:.3f}  RQ {s['rq']:.3f}  P {s['precision']:.3f}  R {s['recall']:.3f}"
              f"  ({int((IMG_NREC == k).sum())} images)")
    if len(GT_DF):
        print("\nGround-truth outcome by size (row %):");            print(pct(pd.crosstab(GT_DF["size"], GT_DF.category, normalize="index")))
        print("\nGround-truth outcome by disk position (row %):");   print(pct(pd.crosstab(GT_DF.radial, GT_DF.category, normalize="index")))
        multi = GT_DF[GT_DF.n_records > 1]
        if len(multi):
            print("\nMulti-annotator images: outcome by how many OTHER annotators drew the same filament (row %, last column = count):")
            t = pd.crosstab(multi.others_who_drew_it.astype(int), multi.category, normalize="index") * 100
            t["filaments"] = multi.groupby(multi.others_who_drew_it.astype(int)).size()
            print(t.round(1).to_string())
        nm = GT_DF[GT_DF.category == "near_miss(0.1-0.5)"].tail_prob.dropna()
        if len(nm):
            print(f"\nNear-misses ({len(nm)}): UNet probability on the annotated pixels the prediction left out -> median {nm.median():.2f};"
                  f" below 0.1: {100 * (nm < 0.1).mean():.0f}%, 0.1-0.4: {100 * ((nm >= 0.1) & (nm < 0.4)).mean():.0f}%, 0.4 or more: {100 * (nm >= 0.4).mean():.0f}%")
        ms = GT_DF[GT_DF.category == "missed(<0.1)"].gt_mean_prob
        if len(ms):
            print(f"Complete misses ({len(ms)}): mean UNet probability inside the filament -> median {ms.median():.2f};"
                  f" below 0.1: {100 * (ms < 0.1).mean():.0f}%")
        print("Unmatched filaments for which the detector's own mask reaches IoU > 0.5 (a mask the fusion could have used):")
        for cat in ("near_miss(0.1-0.5)", "missed(<0.1)", "merged", "fragmented"):
            d = GT_DF[GT_DF.category == cat].detector_best_iou.dropna()
            if len(d):
                print(f"  {cat:20s} {100 * (d > 0.5).mean():5.1f}% of {len(d)}")
    if len(PRED_DF):
        print("\nPredictions by source (counted once per annotator set):"); print(pd.crosstab(PRED_DF.source, PRED_DF.category, margins=True).to_string())
        fp = PRED_DF[(PRED_DF.category != "tp") & (PRED_DF.n_records > 1)]
        if len(fp):
            print(f"False positives on multi-annotator images that match another annotator's filament: {100 * fp.matches_another_annotator.mean():.0f}%")
    ORG_STATS = {"overlapping_pairs": int(len(PAIR_IOU)),
                 "mean_iou": float(PAIR_IOU.mean()) if len(PAIR_IOU) else None,
                 "mean_dice": float(PAIR_DICE.mean()) if len(PAIR_DICE) else None,
                 "gt_overlapping_0_1_2plus": [GT_DEG.get(k, 0) for k in (0, 1, 2)],
                 "pred_overlapping_0_1_2plus": [PRED_DEG.get(k, 0) for k in (0, 1, 2)]}
    write_json(DIAG / "organizer_statistics.json", ORG_STATS)
    print("\nOrganizers' descriptive statistics (out-of-fold):", ORG_STATS)
else:
    ORG_STATS = None
    print("Diagnostics skipped (SFSC_DIAG=0)")


## Cell 14 — Test export for every variant, audit (valid RLE, no empty or overlapping masks)


In [ ]:
LOCAL_EXPORT, EXPORT = LOCAL_ROOT / "test_export", RUN / "test_export"
LOCAL_EXPORT.mkdir(parents=True, exist_ok=True); EXPORT.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = EXPORT / "submission.csv"

def export_task(name):
    """One test image -> candidate features and one RLE per candidate. Scoring happens in the main process."""
    fg, ct = load_maps(*TEST_MAPS[name])
    img, flat = cached_arrays(cache_key("test", name), False)
    lab, n, n_v3, X, area = build_candidates(fg, ct, img, flat, DET_TEST[name])
    rles = [mask_api.encode(np.asfortranarray((lab == i).astype(np.uint8)))["counts"].decode("ascii")
            for i in range(1, n + 1)]
    return name, X, rles

def audit_csv(path, names):
    table = pd.read_csv(path, dtype=str, keep_default_na=False)
    if list(table.columns) != ["filament_id", "segmentation_rle"] or table.filament_id.duplicated().any():
        raise ValueError(f"{path.name}: wrong columns or duplicate filament ids")
    stems = {Path(n).stem for n in names}
    by_stem = defaultdict(list)
    for fid, rle in table.itertuples(index=False):
        stem, suffix = fid.rsplit("_", 1)
        by_stem[stem].append((int(suffix), rle))
    if not set(by_stem) <= stems:
        raise ValueError(f"{path.name}: rows for unknown images")
    for stem, items in by_stem.items():
        if sorted(i for i, _ in items) != list(range(1, len(items) + 1)):
            raise AssertionError(f"{path.name}: ids of {stem} are not 1..n")
        seen = np.zeros((H, W), bool)
        for _, rle in items:
            m = decode_mask({"size": [H, W], "counts": rle})
            if not m.any() or (seen & m).any():
                raise AssertionError(f"{path.name}: empty or overlapping mask in {stem}")
            seen |= m
    return len(table), len(stems) - len(by_stem)

if CFG.export_test:
    test_names = sorted(TEST_MAPS)
    pool = get_pool(refresh=True)                 # workers must see the fusion functions and the test detections
    rows, n_v3, n_det, feat_sum = {v: [] for v in FINAL}, 0, 0, np.zeros(4)
    for name, X, rles in tqdm(pool.map(export_task, test_names, chunksize=1), total=len(test_names), desc="Test export"):
        stem, from_v3 = Path(name).stem, X[:, -1] == 0
        for v, f in FINAL.items():
            u = np.full(len(X), -np.inf)
            sel = np.ones(len(X), bool) if f["all"] else from_v3
            if sel.any():
                u[sel] = f["model"].predict(X[sel][:, f["cols"][0]:f["cols"][1]])
            rows[v] += [(f"{stem}_{j}", rles[i]) for j, i in enumerate(np.flatnonzero(u >= f["tau"]), 1)]
        n_v3 += int(from_v3.sum()); n_det += int((~from_v3).sum())
        feat_sum += X[from_v3][:, [F_IDX["mean_p"], F_IDX["max_p"], N_V3F, N_V3F + 2]].sum(0)
    for v in FINAL:
        df = pd.DataFrame(rows[v], columns=["filament_id", "segmentation_rle"])
        df.to_csv(LOCAL_EXPORT / f"submission_{v}.csv", index=False)
        copy_force(LOCAL_EXPORT / f"submission_{v}.csv", EXPORT / f"submission_{v}.csv")
        n_rows, n_empty = audit_csv(EXPORT / f"submission_{v}.csv", test_names)
        print(f"{v:12s}: {n_rows} filaments, {n_rows / len(test_names):.2f} per image, {n_empty} images with none"
              f" - audit passed{'   <- adopted' if v == ADOPTED else ''}")
    copy_force(EXPORT / f"submission_{ADOPTED}.csv", SUBMISSION_PATH)

    # The scorer was fitted on single-model out-of-fold maps and one detector per image; test uses 4-model averages.
    oof_v3 = CAND_SRC == 0
    cmp = pd.DataFrame({
        "out-of-fold": [oof_v3.sum() / len(EVAL_NAMES), (~oof_v3).sum() / len(EVAL_NAMES),
                        CAND_X[oof_v3, F_IDX["mean_p"]].mean(), CAND_X[oof_v3, F_IDX["max_p"]].mean(),
                        CAND_X[oof_v3, N_V3F].mean(), CAND_X[oof_v3, N_V3F + 2].mean(),
                        (RESULT[ADOPTED]["u"] >= RESULT[ADOPTED]["tau"]).sum() / len(EVAL_NAMES)],
        "test": [n_v3 / len(test_names), n_det / len(test_names), *(feat_sum / max(n_v3, 1)),
                 len(rows[ADOPTED]) / len(test_names)]},
        index=["UNet candidates per image", "detector-only candidates per image", "mean probability", "max probability",
               "detector score", "detector coverage", "kept per image"])
    print("\nOut-of-fold vs test (large differences mean the test maps behave differently from the calibration maps):")
    print(cmp.round(3).to_string())
    print("\nSubmission:", SUBMISSION_PATH)


## Cell 15 — Manifest, push to Drive


In [ ]:
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True)
(RUN / "environment_v5.txt").write_text(freeze.stdout)
_final = SUMMARY.set_index("variant").loc[ADOPTED]
manifest = {"algorithm": "V5 fusion: averaged UNet maps + cross-fitted YOLO11 detector evidence, calibrated scorer",
            "maps": MAP_RUNS, "aug": AUG,
            "adopted": ADOPTED, "baseline": BASELINE,
            "oof": {k: (float(_final[k]) if k not in ("tp", "fp", "fn") else int(_final[k]))
                    for k in ("tau", "pq", "sq", "rq", "precision", "recall", "tp", "fp", "fn", "gain", "ci_low", "ci_high")},
            "selection": SUMMARY.to_dict("records"), "organizer_statistics": ORG_STATS,
            "v3_run": str(V3_RUN), "v3_recipe_sha256": V3_FROZEN["sha256"], "v3_geometry": GEOM,
            "detector": {"weights": CFG.det_weights, "imgsz": CFG.det_imgsz, "epochs": CFG.det_epochs,
                         "cls_gain": CFG.det_cls_gain, "folds": DET_FOLDS},
            "config": asdict(CFG), "images_scored": len(EVAL_NAMES), "annotator_sets_without_filaments": N_EMPTY_RECORDS,
            "python": platform.python_version(),
            "submission": str(SUBMISSION_PATH) if CFG.export_test else None,
            "submission_sha256": hash_file(SUBMISSION_PATH) if CFG.export_test and SUBMISSION_PATH.exists() else None}
write_json(RUN / "manifest_v5.json", manifest)
push()
print(f"Adopted: {ADOPTED} | out-of-fold PQ {_final.pq:.4f} (SQ {_final.sq:.3f}, RQ {_final.rq:.3f})"
      f" | gain over re-scored V3 {_final.gain:+.4f} (95% interval {_final.ci_low:+.4f} to {_final.ci_high:+.4f})")
print("Artifacts:", RUN)
